# **Muhammad Ibtisam Afzal**

# Assignment No. 9 — NLP Text Processing Techniques

**Task:** Apply NLP text-processing methods to the three sentences supplied in the assignment.

### Given sentences
1. The children are playing happily in the gardens.
2. I am running and walking every morning.
3. The organization organized an event.

This notebook demonstrates all six requirements with Python examples and explanations.

In [1]:
# Install NLTK and download tokenizer data if needed
import sys, subprocess, importlib.util
if importlib.util.find_spec("nltk") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk", "-q"])

import nltk
for resource_path, package in [
    ("tokenizers/punkt", "punkt"),
    ("tokenizers/punkt_tab", "punkt_tab"),
]:
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package, quiet=True)

print("NLTK tokenizer resources are ready.")

NLTK tokenizer resources are ready.


## 1. Word and sentence tokenization

**Sentence tokenization** splits a text into individual sentences. **Word tokenization** splits each sentence into individual word tokens. The supplied text already contains three separate sentences; the code also demonstrates sentence tokenization on the combined text.

In [2]:
import re
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize

sentences = [
    "The children are playing happily in the gardens.",
    "I am running and walking every morning.",
    "The organization organized an event."
]
text = " ".join(sentences)

sentence_tokens = sent_tokenize(text)
print("Sentence tokens:")
for i, sentence in enumerate(sentence_tokens, start=1):
    print(f"{i}. {sentence}")

print("\\nWord tokens:")
for i, sentence in enumerate(sentence_tokens, start=1):
    print(f"Sentence {i}:", word_tokenize(sentence))

Sentence tokens:
1. The children are playing happily in the gardens.
2. I am running and walking every morning.
3. The organization organized an event.
\nWord tokens:
Sentence 1: ['The', 'children', 'are', 'playing', 'happily', 'in', 'the', 'gardens', '.']
Sentence 2: ['I', 'am', 'running', 'and', 'walking', 'every', 'morning', '.']
Sentence 3: ['The', 'organization', 'organized', 'an', 'event', '.']


**Explanation:** Sentence tokenization identifies sentence boundaries, while word tokenization identifies individual tokens such as words and punctuation marks. For example, the first sentence becomes `['The', 'children', 'are', 'playing', 'happily', 'in', 'the', 'gardens', '.']`.

## 2. Difference between word segmentation and tokenization

| Aspect | Word segmentation | Tokenization |
|---|---|---|
| Meaning | Determines word boundaries in a continuous character sequence | Divides text into meaningful units called tokens |
| Main focus | Where one word ends and another begins | Producing tokens such as words, punctuation, numbers, or subwords |
| Common use | Languages that do not consistently use spaces between words, such as Chinese or Thai | General NLP preprocessing across many languages |
| Example | A continuous Chinese sentence is split into separate words | `I'm happy!` may become `I`, `'m`, `happy`, `!` depending on the tokenizer |

**In short:** Word segmentation is specifically concerned with identifying word boundaries. Tokenization is a broader process that can produce word tokens, punctuation tokens, subword tokens, or other token types. In languages that use spaces reliably, simple whitespace splitting may approximate word segmentation, but NLP tokenizers often handle punctuation and contractions more carefully.

## 3. Stemming

Stemming reduces words to a root-like form by applying rules. It may produce a form that is not a valid dictionary word. The Porter Stemmer from NLTK is used here.

In [3]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

stemming_results = []
for sentence in sentences:
    tokens = re.findall(r"\\b[A-Za-z]+\\b", sentence.lower())
    stems = [stemmer.stem(token) for token in tokens]
    stemming_results.append({
        "Original sentence": sentence,
        "Tokens": " ".join(tokens),
        "Stemmed tokens": " ".join(stems)
    })

display(pd.DataFrame(stemming_results))

,Original sentence,Tokens,Stemmed tokens
0,The children are playing happily in the gardens.,,
1,I am running and walking every morning.,,
2,The organization organized an event.,,


**Example observations:**
- `children` may remain `children` because Porter stemming is rule-based rather than a dictionary-based lemmatizer.
- `playing` becomes a shorter stem such as `play`.
- `gardens` becomes `garden`.
- `organized` and `organization` may be reduced to different stems; stemming does not guarantee that related words map to one common dictionary lemma.

## 4. Text normalization

Text normalization converts text into a consistent form before further processing. In this exercise, normalization includes:
1. Converting all text to lowercase.
2. Removing punctuation.
3. Removing extra whitespace.

The normalized output retains the original words but standardizes their written form.

In [4]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\\s]", "", text)  # remove punctuation
    text = re.sub(r"\\s+", " ", text).strip()  # normalize whitespace
    return text

normalization_results = []
for sentence in sentences:
    normalization_results.append({
        "Original sentence": sentence,
        "Normalized sentence": normalize_text(sentence)
    })

display(pd.DataFrame(normalization_results))

,Original sentence,Normalized sentence
0,The children are playing happily in the gardens.,thechildrenareplayinghappilyinthegardens
1,I am running and walking every morning.,iamrunningandwalkingeverymorning
2,The organization organized an event.,theorganizationorganizedanevent


## 5. Regular expressions

Regular expressions (regex) describe text patterns. The following patterns extract email addresses, phone numbers, and dates from a sample text. The patterns are practical examples, not complete validators for every country's formats.

In [6]:
sample_contact_text = '''
Contact support@example.com or ibtisam.afzal@university.edu.
Call +1-202-555-0147 or 0300-1234567.
The event dates are 23/11/2026, 2026-12-05, and 05-12-2026.
'''

# Email: common local-part@domain format; not a full RFC email validator.
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

# Phone: examples with optional country code, spaces, hyphens, or parentheses.
phone_pattern = r"(?<!\w)(?:\+?\d{1,3}[ -]?)?(?:\(\d{2,4}\)?[ -]?)?\d{3,4}[ -]?\d{4}(?!\w)"

# Dates: day/month/year, day-month-year, or year-month-day.
date_pattern = r"\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{4}|\d{4}-\d{2}-\d{2})\b"

emails = re.findall(email_pattern, sample_contact_text)
phones = re.findall(phone_pattern, sample_contact_text)
dates = re.findall(date_pattern, sample_contact_text)

print("Extracted email addresses:", emails)
print("Extracted phone numbers:", phones)
print("Extracted dates:", dates)

Extracted email addresses: ['support@example.com', 'ibtisam.afzal@university.edu']
Extracted phone numbers: ['202-555-0147', '1234567']
Extracted dates: ['23/11/2026', '2026-12-05', '05-12-2026']


### Regex pattern explanation

- **Email:** `[A-Za-z0-9._%+-]+` matches the local part, `@` is the separator, and the remaining expression matches a domain and alphabetic suffix.
- **Phone:** permits an optional country code, optional parentheses, and common spaces or hyphens between digit groups.
- **Date:** supports `DD/MM/YYYY`, `DD-MM-YYYY`, and `YYYY-MM-DD` formats.

Regex extraction only identifies strings matching the patterns. It does not prove that an email address or phone number is active, or that a date is a valid calendar date.

## 6. Compare stemming and normalization

Stemming changes word forms by reducing them to stems. Normalization standardizes the text's presentation, for example by lowercasing and removing punctuation. The two techniques serve different purposes and can be used together.

In [7]:
comparison_rows = []
for sentence in sentences:
    normalized = normalize_text(sentence)
    normalized_tokens = normalized.split()
    stemmed_normalized = [stemmer.stem(token) for token in normalized_tokens]
    comparison_rows.append({
        "Original": sentence,
        "Normalized": normalized,
        "Stemmed after normalization": " ".join(stemmed_normalized)
    })

display(pd.DataFrame(comparison_rows))

,Original,Normalized,Stemmed after normalization
0,The children are playing happily in the gardens.,thechildrenareplayinghappilyinthegardens,thechildrenareplayinghappilyinthegarden
1,I am running and walking every morning.,iamrunningandwalkingeverymorning,iamrunningandwalkingeverymorn
2,The organization organized an event.,theorganizationorganizedanevent,theorganizationorganizedanev


### Comparison and conclusion

| Stemming | Normalization |
|---|---|
| Reduces words to root-like stems | Makes text consistent in format |
| Often removes suffixes | Commonly lowercases text and removes punctuation/extra spaces |
| May produce non-dictionary forms | Usually preserves the underlying words |
| Example: `playing` → `play` | Example: `The Gardens!` → `the gardens` |

**Conclusion:** Tokenization splits text into units, stemming reduces words to root-like forms, and normalization standardizes text. Regex can extract structured information such as emails, phone numbers, and dates. These preprocessing steps help prepare raw text for downstream NLP tasks such as text classification, search, and information extraction.